# Prompt Engineering Techniques

This notebook demonstrates **11 prompt engineering techniques** using the **Ollama LLaMA 3.2:3B** model locally.

Each technique is presented in its own cell with an explanation and working example.

In [1]:
# Cell 1: Imports
import ollama
import json

In [2]:
# Cell 2: Model Configuration

MODEL = "llama3.2:3b"

# Test the connection
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(f"Model: {MODEL}")
print(f"Test Response: {response['message']['content']}")

Model: llama3.2:3b
Test Response: Hello!


In [3]:
print(response)

model='llama3.2:3b' created_at='2026-04-13T15:10:51.0231032Z' done=True done_reason='stop' total_duration=8336189300 load_duration=8002781600 prompt_eval_count=459 prompt_eval_duration=285744900 eval_count=3 eval_duration=42158800 message=Message(role='assistant', content='Hello!', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


## 1. Zero-Shot Prompting

The simplest technique -- ask the model to perform a task **without providing any examples**. The model relies entirely on its pre-trained knowledge.

**When to use:** Simple, well-defined tasks where the model likely already understands the expected output format.

In [4]:
# Technique 1: Zero-Shot Prompting
# No examples provided -- the model uses only its pre-trained knowledge

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": """Classify the sentiment of the following movie review as Positive, Negative, or Neutral.

Review: The cinematography was breathtaking and the storyline kept me on the edge of my seat the entire time.

Sentiment:"""
        }
    ]
)
print(response)
print(response["message"]["content"])

model='llama3.2:3b' created_at='2026-04-13T15:13:36.9124826Z' done=True done_reason='stop' total_duration=2079445900 load_duration=249200400 prompt_eval_count=495 prompt_eval_duration=202502800 eval_count=91 eval_duration=1573969000 message=Message(role='assistant', content='Based on the review, I would classify the sentiment as **Positive**.\n\nThe reviewer uses the following positive language:\n\n* "breathtaking" to describe the cinematography, which is a strong positive adjective\n* "kept me on the edge of my seat" to describe the storyline, which suggests that the reviewer was engaged and excited by the movie\n\nThere is no negative language or criticism in the review, which further supports the classification as Positive.', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
Based on the review, I would classify the sentiment as **Positive**.

The reviewer uses the following positive language:

* "breathtaking" to describe the cinematography, which is a stro

## 2. Few-Shot Prompting

Provide the model with a **few examples** of the desired input-output behavior before asking it to perform the task. This guides the model toward the expected format and logic.

**When to use:** When zero-shot results are inconsistent or when you need a very specific output format.

In [4]:
# Technique 2: Few-Shot Prompting
# Provide examples to guide the model's output format and reasoning

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": """Classify the following texts into categories: Sports, Technology, or Politics.

Text: The new iPhone 16 features a revolutionary AI chip.
Category: Technology

Text: The senator proposed a new healthcare reform bill.
Category: Politics

Text: The team won the championship after a thrilling overtime.
Category: Sports

Text: Researchers developed a quantum computing breakthrough that could change encryption.
Category:"""
        }
    ]
)

print(response["message"]["content"])

Based on the text, I would classify it as:

Category: Technology

The mention of "quantum computing breakthrough" and the comparison to "AI chip" suggests that the text is related to technological advancements.

So, the updated classification is:

Text: Researchers developed a quantum computing breakthrough that could change encryption.
Category: Technology


## 3. Role Prompting

Assign the model a **specific persona or role** using the system message. This shapes the tone, vocabulary, depth, and perspective of the response.

**When to use:** When you need domain-specific expertise, a particular communication style, or want responses tailored to a specific audience.

In [6]:
# Technique 3: Role Prompting
# Assign a persona via the system message to shape the response

# Role 1: Senior data scientist
response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a senior data scientist with 15 years of experience. You explain concepts clearly with real-world examples and best practices."
        },
        {
            "role": "user",
            "content": "What is overfitting in machine learning and how do I prevent it?"
        }
    ]
)

print("--- Role: Senior Data Scientist ---")
print(response["message"]["content"])

# Role 2: Teacher for kids
print("\n" + "=" * 50)

response_2 = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a friendly teacher explaining concepts to a 10-year-old. Use simple words and fun analogies."
        },
        {
            "role": "user",
            "content": "What is overfitting in machine learning and how do I prevent it?"
        }
    ]
)

print("\n--- Role: Teacher for Kids ---")
print(response_2["message"]["content"])

--- Role: Senior Data Scientist ---
Overfitting is a common problem in machine learning where a model is too complex and fits the noise in the training data too closely, resulting in poor performance on unseen data. In other words, the model becomes too specialized to the training data and fails to generalize well to new, unseen data.

**What causes overfitting?**

There are several reasons why overfitting occurs:

1. **Model complexity**: When the model is too complex, it may start to fit the noise in the training data rather than the underlying patterns.
2. **Insufficient training data**: If the training data is too small, the model may not have enough examples to learn from, leading to overfitting.
3. **Regularization**: Lack of regularization techniques can lead to overfitting, as the model is not penalized enough for fitting the noise.
4. **Optimization algorithm**: Some optimization algorithms can lead to overfitting, especially if they are not well-tuned.

**Signs of overfitting

## 4. Structured Output

Instruct the model to return its response in a **specific format** such as JSON, CSV, Markdown tables, or XML. This makes the output machine-readable and easy to parse programmatically.

**When to use:** When downstream code needs to consume the model's output, or when you need consistent, parseable responses.

In [7]:
# Technique 4: Structured Output
# Request the model to respond in a specific parseable format (JSON)

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": """Extract the following information from the text below and return it as a JSON object with these keys:
- name, age, occupation, skills (as a list), location

Text: "John Smith is a 32-year-old software engineer based in San Francisco.
He is proficient in Python, JavaScript, and cloud computing, and recently started learning Rust."

Return ONLY valid JSON, no explanation."""
        }
    ]
)

result = response["message"]["content"]
print("Raw Response:")
print(result)

# Try to parse the JSON
try:
    parsed = json.loads(result)
    print("\nParsed JSON successfully!")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print("\nNote: Response was not pure JSON -- may need prompt refinement")

Raw Response:
{"name": "John Smith", "age": 32, "occupation": "software engineer", "skills": ["Python", "JavaScript", "cloud computing", "Rust"], "location": "San Francisco"}

Parsed JSON successfully!
{
  "name": "John Smith",
  "age": 32,
  "occupation": "software engineer",
  "skills": [
    "Python",
    "JavaScript",
    "cloud computing",
    "Rust"
  ],
  "location": "San Francisco"
}


## 5. Prompt Template

Create a **reusable template** with placeholders that can be filled in dynamically. This standardizes prompts across different inputs and makes them easy to maintain.

**When to use:** When you run the same type of prompt repeatedly with different inputs (e.g., summarization, translation, analysis).

In [6]:
# Technique 5: Prompt Template
# Create reusable templates with placeholders for dynamic inputs

# Define a reusable template
summary_template = """
Summarize the following {doc_type} in {num_sentences} sentences.
Target audience: {audience}
Tone: {tone}

Text:
{text}

Summary:"""

# --- Use 1: Research abstract for students ---
prompt_1 = summary_template.format(
    doc_type="research abstract",
    num_sentences="2",
    audience="undergraduate students",
    tone="informative and accessible",
    text="Large Language Models (LLMs) have demonstrated remarkable capabilities across various "
         "natural language processing tasks. Recent research has shown that techniques such as "
         "chain-of-thought prompting, few-shot learning, and reinforcement learning from human "
         "feedback (RLHF) significantly improve model performance. However, challenges remain in "
         "areas such as hallucination reduction, factual grounding, and computational efficiency."
)

print(prompt_1)

response_1 = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_1}]
)

print("--- Use 1: Research abstract for students ---")
print(response_1["message"]["content"])

# --- Use 2: News article for executives ---
print("\n" + "=" * 50)

prompt_2 = summary_template.format(
    doc_type="news article",
    num_sentences="1",
    audience="busy executives",
    tone="concise and direct",
    text="Apple announced its latest lineup of products at the annual WWDC event, including a "
         "new AI-powered Siri, updates to macOS, iOS, and a surprise reveal of AR glasses "
         "that integrate with the Vision Pro ecosystem."
)

response_2 = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_2}]
)

print("\n--- Use 2: News article for executives ---")
print(response_2["message"]["content"])


Summarize the following research abstract in 2 sentences.
Target audience: undergraduate students
Tone: informative and accessible

Text:
Large Language Models (LLMs) have demonstrated remarkable capabilities across various natural language processing tasks. Recent research has shown that techniques such as chain-of-thought prompting, few-shot learning, and reinforcement learning from human feedback (RLHF) significantly improve model performance. However, challenges remain in areas such as hallucination reduction, factual grounding, and computational efficiency.

Summary:
--- Use 1: Research abstract for students ---
Here's a summary of the research abstract in 2 sentences:

Large Language Models (LLMs) have shown impressive abilities in natural language processing tasks, but there are still challenges to overcome, such as producing accurate and reliable results and using them efficiently. Researchers are working to address these challenges through techniques like chain-of-thought pro

## 6. Chain of Thought (CoT)

Ask the model to **think step-by-step** before arriving at a final answer. This dramatically improves performance on reasoning, math, and logic tasks.

**When to use:** Math problems, logic puzzles, multi-step reasoning, or any task where intermediate steps improve accuracy.

In [10]:
# Technique 6: Chain of Thought (CoT)
# Ask the model to reason step-by-step before answering

question = """A farmer has 3 fields. Field A produces twice as much wheat as Field B.
Field C produces 30% more than Field A. Together, all three fields produce 1,560 kg of wheat.
The farmer sells wheat at $4/kg but pays 15% tax on the revenue.
After tax, he splits the profit equally among his 4 workers.
How much does each worker get?"""

# WITHOUT Chain of Thought
response_no_cot = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": question}]
)

print("--- WITHOUT Chain of Thought ---")
print(response_no_cot["message"]["content"])

# WITH Chain of Thought
print("\n" + "=" * 50)

response_cot = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": question + "\n\nLet's solve this step by step:"
        }
    ]
)

print("\n--- WITH Chain of Thought ---")
print(response_cot["message"]["content"])

--- WITHOUT Chain of Thought ---
Let's break down the problem step by step:

1. Let's say Field B produces x kg of wheat. Then, Field A produces 2x kg of wheat (since it produces twice as much as Field B).
2. Field C produces 30% more than Field A, so it produces 2x + 0.3(2x) = 2.6x kg of wheat.
3. The total amount of wheat produced by all three fields is x + 2x + 2.6x = 5.6x kg.
4. We know that the total amount of wheat produced is 1,560 kg, so we can set up the equation: 5.6x = 1,560.
5. Solving for x, we get x = 1,560 / 5.6 = 280 kg (this is the amount of wheat produced by Field B).
6. Now, we can find the amount of wheat produced by Field A and Field C: 2x = 2(280) = 560 kg and 2.6x = 2.6(


--- WITH Chain of Thought ---
Let's break down the problem step by step.

We know that Field A produces twice as much wheat as Field B, so we can represent the amount of wheat produced by Field B as x. Therefore, the amount of wheat produced by Field A is 2x.

We also know that Field C produces

## 7. Tree of Thought (ToT)

Explore **multiple reasoning paths** simultaneously and evaluate which one leads to the best answer. The model considers different approaches, assesses their merit, and selects the strongest.

**When to use:** Complex problems with multiple possible approaches, creative tasks, or decisions that benefit from considering alternatives.

In [11]:
# Technique 7: Tree of Thought (ToT)
# Explore multiple reasoning paths and evaluate the best one

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": """I want to improve the performance of my Python web application.

Consider 3 different expert approaches to this problem:

Expert 1 (Backend Optimization): Think about this from a server-side perspective.
- What is your proposed approach?
- What are the pros and cons?

Expert 2 (Database Optimization): Think about this from a data layer perspective.
- What is your proposed approach?
- What are the pros and cons?

Expert 3 (Architecture): Think about this from a system design perspective.
- What is your proposed approach?
- What are the pros and cons?

Finally, synthesize the best ideas from all three experts into a unified recommendation."""
        }
    ]
)

print(response["message"]["content"])

Here are the proposed approaches from each of the three experts:

**Expert 1: Backend Optimization**

Proposed Approach: Implement a caching mechanism for frequently accessed data, use a content delivery network (CDN) to reduce latency, and consider using a load balancer to distribute traffic across multiple servers.

Pros:

* Improved performance for frequent requests
* Reduced load on servers
* Better scalability

Cons:

* Additional complexity and maintenance overhead
* May not be suitable for all types of requests (e.g., complex database queries)

To implement caching, we can use libraries like `cachetools` or ` Redis` to store frequently accessed data. We can also use a CDN like `Cloudflare` to reduce latency. For load balancing, we can use a library like `HAProxy` or `NGINX`.

**Expert 2: Database Optimization**

Proposed Approach: Optimize database queries using indexing, partitioning, and query optimization. Consider using a database like PostgreSQL or MongoDB, which are known 

## 8. Meta Prompting

Use the model to **generate or improve prompts** for itself. This is "prompting about prompting" -- you ask the model to craft an optimal prompt for a given task.

**When to use:** When you're unsure how to best phrase a prompt, or when you want the model to help optimize its own instructions.

In [12]:
# Technique 8: Meta Prompting
# Use the model to generate or improve prompts

# Step 1: Ask the model to CREATE an optimized prompt
response_meta = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": """I want to use an LLM to analyze customer support tickets and extract:
- The main issue
- Urgency level (low, medium, high)
- Suggested department to handle it

Write an optimized prompt that I can use for this task.
The prompt should be clear, specific, and produce consistent structured output."""
        }
    ]
)

generated_prompt = response_meta["message"]["content"]
print("--- Step 1: Model-Generated Prompt ---")
print(generated_prompt)

# Step 2: USE the generated prompt on a real ticket
print("\n" + "=" * 50)
print("\n--- Step 2: Using the generated prompt ---\n")

test_ticket = """Customer says: 'My payment was charged twice for order #4521.
I need an immediate refund for the duplicate charge.
This is the third time this has happened and I'm considering canceling my account.'"""

response_use = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": generated_prompt + "\n\nCustomer Ticket: " + test_ticket
        }
    ]
)

print(response_use["message"]["content"])

--- Step 1: Model-Generated Prompt ---
Here's an optimized prompt that you can use for analyzing customer support tickets with an LLM:

**Prompt:**

"I'll provide a sample customer support ticket. Please respond in the following format:

**Main Issue:** [insert relevant keyword or phrase extracted from the ticket]

**Urgency Level:** [insert one of the following labels: Low, Medium, High]

**Suggested Department:** [insert a department name, e.g. "IT Support", "Customer Service", etc.]

Example output format:

Main Issue: [insert keyword or phrase]
Urgency Level: [insert label]
Suggested Department: [insert department name]

Please use the following ticket as an example:
[Ticket text]

Please respond in the requested format.

**Additional constraints:**

* Assume that the ticket is written in standard English and does not contain any technical jargon or specialized terminology.
* Assume that the ticket is a typical example of a customer support query, with a clear and concise issue des

## 9. ReAct Pattern (Reasoning + Acting)

Combine **reasoning** (Thought) with **acting** (Action) and **observing** (Observation) in an interleaved loop. The model thinks about what to do, takes an action, observes the result, and iterates.

**When to use:** Tasks that require interacting with external tools, APIs, or environments -- the foundation of LLM agents.

In [17]:
# Technique 9: ReAct Pattern (Reasoning + Acting)
# Simulate the Thought -> Action -> Observation loop

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": """You solve problems using the ReAct framework.
For EACH step, output exactly these three lines:
Thought: (your reasoning about what to do next)
Action: (one of: Calculate, Search, Lookup, Finish)
Observation: (the result of your action)

Keep going step by step until you reach the final answer"""
        },
        {
            "role": "user",
            "content": """A company had 120 employees in January.
They hired 15% more in Q1, then lost 10 employees in Q2, then hired 20 more in Q3.
What is the total headcount at the end of Q3, and what is the percentage increase from January?

Thought 1:"""
        }
    ]
)

print(response["message"]["content"])

Action: Calculate

Observation: 
January headcount = 120 
15% of 120 = 18, so they hired 18 more in Q1, so total headcount in Q1 = 120 + 18 = 138
Then they lost 10 employees in Q2, so total headcount in Q2 = 138 - 10 = 128
Then they hired 20 more in Q3, so total headcount in Q3 = 128 + 20 = 148


## 10. Self-Consistency

Generate **multiple independent responses** to the same prompt and then determine the most consistent (majority) answer. This reduces the impact of any single flawed reasoning chain.

**When to use:** High-stakes reasoning tasks where accuracy matters more than speed -- math, logic, factual questions.

In [18]:
# Technique 10: Self-Consistency
# Generate multiple responses and pick the majority answer

question = """If a shirt originally costs $60 and is on sale for 25% off,
then you apply an additional 10% loyalty discount on the sale price,
how much do you pay?

Think step by step and give the final price."""

NUM_SAMPLES = 3
responses = []

print(f"Generating {NUM_SAMPLES} independent responses...\n")

for i in range(NUM_SAMPLES):
    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": question}]
    )
    answer = resp["message"]["content"]
    responses.append(answer)
    print(f"--- Response {i + 1} ---")
    print(answer)
    print()

# Ask the model to determine the consensus
print("=" * 50)
print("\n--- Determining Consensus ---\n")

consensus_input = "I asked the same math question multiple times and got these responses:\n\n"
for i, r in enumerate(responses):
    consensus_input += f"Response {i + 1}: {r}\n\n"
consensus_input += "What is the most common final answer among these responses? State just the final price."

consensus = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": consensus_input}]
)

print(f"Consensus Answer: {consensus['message']['content']}")

Generating 3 independent responses...

--- Response 1 ---
Let's break it down step by step:

1. First, we calculate the discount amount:
Original price: $60
Discount percentage: 25%
Discount amount: $60 x 25% = $60 x 0.25 = $15

Sale price (after 25% discount): $60 - $15 = $45

2. Next, we apply the additional 10% loyalty discount on the sale price:
Loyalty discount percentage: 10%
Loyalty discount amount: $45 x 10% = $45 x 0.10 = $4.50

Final sale price (after both discounts): $45 - $4.50 = $40.50

So, after applying both discounts, the final price you pay for the shirt is $40.50.

--- Response 2 ---
To find the final price, we need to apply the discounts step by step. Here's the calculation:

1. First, we apply the 25% discount on the original price of $60:
25% of $60 = 0.25 x $60 = $15
Discounted price = $60 - $15 = $45

2. Then, we apply the 10% loyalty discount on the sale price of $45:
10% of $45 = 0.10 x $45 = $4.50
Final discounted price = $45 - $4.50 = $40.50

Therefore, the f

## 11. Prompt Chaining

Break a complex task into **multiple sequential steps**, where the output of one prompt becomes the input to the next. Each step handles one focused sub-task.

**When to use:** Complex multi-step tasks, long-form content generation, analysis pipelines, or when a single prompt would be too complex.

In [20]:
# Technique 11: Prompt Chaining
# Break complex tasks into sequential steps, feeding output into the next prompt
# Example: App Ideation Pipeline

app_idea = "A mobile app that helps college students manage their mental health and study habits"

# ---- CHAIN STEP 1: Generate core features ----
print("=== CHAIN STEP 1: Generate Core Features ===")

step1 = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""I want to build: {app_idea}

List exactly 5 core features this app should have.
Return only the features as a numbered list."""
        }
    ]
)

features = step1["message"]["content"]
print(features)

# ---- CHAIN STEP 2: Define user stories for each feature ----
print("=== CHAIN STEP 2: User Stories ===")

step2 = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""For the following app features, write one user story per feature
in the format: "As a [user], I want to [action], so that [benefit]."

App: {app_idea}

Features:
{features}

User Stories:"""
        }
    ]
)

user_stories = step2["message"]["content"]
print(user_stories)

# ---- CHAIN STEP 3: Tech stack recommendation ----
print("=== CHAIN STEP 3: Tech Stack Recommendation ===")

step3 = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""Based on these features and user stories for a mobile app:

App: {app_idea}

Features:
{features}

User Stories:
{user_stories}

Recommend a tech stack (frontend, backend, database, and any APIs)
and briefly justify each choice in one sentence."""
        }
    ]
)

tech_stack = step3["message"]["content"]
print(tech_stack)

# ---- CHAIN STEP 4: MVP scope and timeline ----
print("=== CHAIN STEP 4: MVP Scope & Timeline ===")

step4 = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""Given this app plan:

App: {app_idea}
Features: {features}
Tech Stack: {tech_stack}

Define an MVP by selecting the top 3 features to build first.
For each, estimate the development effort (small/medium/large)
and suggest a realistic timeline for a team of 2 developers."""
        }
    ]
)

print(step4["message"]["content"])


=== CHAIN STEP 1: Generate Core Features ===
1. Mood Tracker: A daily mood journal that allows users to log their emotions, thoughts, and physical sensations, with the option to add tags and notes for later analysis.

2. Study Planner: A calendar-based planner that helps users set study goals, schedule regular study sessions, and set reminders for upcoming deadlines.

3. Habit Tracker: A habit tracker that allows users to set and track daily habits related to mental health, such as meditation, exercise, or sleep, with rewards and motivational messages.

4. Resource Library: A database of mental health resources, including articles, videos, and podcasts, curated by mental health professionals and organized by topic.

5. Community Forum: A private online forum where users can connect with peers who share similar interests, goals, or struggles, facilitated by moderation and strict guidelines to ensure a safe and supportive environment.
=== CHAIN STEP 2: User Stories ===
Here are the user 